# Experimentos de baseline

Executa B0, M1, M2 e M3 sem tocar no conjunto de teste durante a selecao de regularizacao. Consulte `docs/05-hipoteses-de-modelagem.md` para as hipoteses e criterios de decisao.

In [1]:
from pathlib import Path
import runpy

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
runpy.run_path(ROOT / 'scripts' / 'run_baseline_experiments.py', run_name='__main__')

usage: run_baseline_experiments.py [-h]
                                   [--feature-set {baseline,text_metadata,selected_text,historical,selected_history}]
                                   [--output-dir OUTPUT_DIR]
run_baseline_experiments.py: error: argument --feature-set: invalid choice: 'c:\\Users\\Inteli\\AppData\\Roaming\\jupyter\\runtime\\kernel-v3a033493c6e7ec8ed6101cf8fca00d0f0a3d7cdc7.json' (choose from baseline, text_metadata, selected_text, historical, selected_history)


SystemExit: 2

c:\Users\Inteli\Documents\Projeto_Inteli\case_galaxies\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3831: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
import json
import pandas as pd

ARTIFACTS = ROOT / 'artifacts' / 'baseline'
summary = pd.read_csv(ARTIFACTS / 'metrics_summary.csv')
statistics = json.loads((ARTIFACTS / 'comparison_statistics.json').read_text(encoding='utf-8'))
summary_display = summary.copy()
for column in ['macro_mae_log_likes', 'macro_rmse_log_likes', 'macro_bias_log_likes']:
    summary_display[column] = summary_display[column].map(lambda value: f'{value:.4f}')
summary_display['macro_pairwise_accuracy'] = summary_display['macro_pairwise_accuracy'].map(lambda value: f'{value:.1%}')
display(summary_display)
statistics

## Leitura correta

A escolha nao deve depender apenas do menor valor da tabela. Para cada comparacao, verificamos a direcao do ganho, o intervalo bootstrap por canal e o teste de permutacao. Se M2 nao apresentar vantagem consistente sobre M1, M1 continua sendo a arquitetura preferida por ser mais simples e usar todos os dados de treino.

In [ ]:
per_channel = pd.read_csv(ARTIFACTS / 'per_channel_metrics.csv')
comparison = (
    per_channel.pivot(index=['channel_name', 'channel_type'], columns='model', values='mae_log_likes')
    .assign(
        m1_minus_b0=lambda frame: frame['M1'] - frame['B0'],
        m2_minus_m1=lambda frame: frame['M2'] - frame['M1'],
        m1_minus_m3=lambda frame: frame['M1'] - frame['M3'],
    )
    .sort_values('m2_minus_m1')
)
display(comparison.round(4))